In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [14]:
df = pd.read_csv('AQI.csv')

TASK 1:

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1396 entries, 0 to 1395
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        1396 non-null   object 
 1   state          1396 non-null   object 
 2   city           1395 non-null   object 
 3   date_time      1396 non-null   object 
 4   latitude       1381 non-null   float64
 5   longitude      1381 non-null   float64
 6   pollutant_id   1396 non-null   object 
 7   pollutant_avg  1295 non-null   float64
dtypes: float64(3), object(5)
memory usage: 87.4+ KB


info() gives information: name of columns, number of entries, number of non null cells and data type of each column about the dataframe 

the non null count gives the number of rows that have something in them, i.e. aren't null. 

In [4]:
df.describe()

,latitude,longitude,pollutant_avg
count,1381.000000,1381.000000,1295.000000
mean,22.187413,78.641514,35.006950
std,5.589081,4.872921,26.605727
min,8.514909,70.909168,1.000000
25%,18.976700,75.387310,16.500000
50%,22.968259,77.400574,29.000000
75%,26.803650,80.612222,46.000000
max,34.066206,94.636574,263.000000


describe() gives all statistics about the numeric columns of the dataframe

In [5]:
df.head()

,country,state,city,date_time,latitude,longitude,pollutant_id,pollutant_avg
0,India,Andhra_Pradesh,Tirumala,14-08-2025 12:00,13.670000,79.350000,PM2.5,15.0
1,India,Andhra_Pradesh,Tirumala,14-08-2025 12:00,13.670000,79.350000,O3,13.0
2,India,Andhra_Pradesh,Tirupati,14-08-2025 12:00,13.615387,79.409230,PM2.5,NaN
3,India,Andhra_Pradesh,Vijayawada,14-08-2025 12:00,16.536107,80.594233,PM10,73.0
4,India,Arunachal Pradesh,Naharlagun,14-08-2025 12:00,27.103358,93.679645,PM10,NaN


head() gives the top 5 entries of the data frame

In [7]:
df.shape

(1396, 8)

.shape gives the numbers of rows and columns of the dataframe

TASK 2:

In [11]:
print("Number of null values in each column: ")
print("countries:", df['country'].isnull().sum())
print("states:", df['state'].isnull().sum())
print("cities:", df['city'].isnull().sum())
print("date_time:", df['date_time'].isnull().sum())
print("latitude:", df['latitude'].isnull().sum())
print("longitude:", df['longitude'].isnull().sum())
print("polutant_id:", df['pollutant_id'].isnull().sum())
print("pollutant_avg:", df['pollutant_avg'].isnull().sum())

Number of null values in each column: 
countries: 0
states: 0
cities: 1
date_time: 0
latitude: 15
longitude: 15
polutant_id: 0
pollutant_avg: 101


In [16]:
df = df.dropna(subset=['city'])

In [17]:
print("cities:", df['city'].isnull().sum())

cities: 0


earlier, there was one missing value in city column but now it has been dropped

In [ ]:
# Make a mapping: city -> (latitude, longitude)
city_coords = df.groupby('city')[['latitude', 'longitude']].first().to_dict('index')

# Fill missing latitude/longitude
df['latitude'] = df.apply(
    lambda row: city_coords[row['city']]['latitude'] 
    if pd.isna(row['latitude']) 
    else row['latitude'],
    axis=1
)

df['longitude'] = df.apply(
    lambda row: city_coords[row['city']]['longitude'] 
    if pd.isna(row['longitude']) 
    else row['longitude'],
    axis=1
)


In [19]:
print("latitude:", df['latitude'].isnull().sum())
print("longitude:", df['longitude'].isnull().sum())

latitude: 0
longitude: 0


here i first created a dictionary mapping the cities to their latitude and longitude coordinates. i used groupby() to group cities and find the first available coordinate set and added it to the dictionary.

first() takes one representattive row per city and to_dict() changed it to a dictionary.

using apply() function, i run a custom function, passing each row in the lambda function as a Series.

then inside the lambda function check if pd.isna(row['longitude/latitude']) is true or not; if it is true then go to the dictionary and adds the latitude matched with the city, else keeps the same latitude or longitude.

axis = 1 means checking rows.

In [22]:
df.groupby('pollutant_id')['pollutant_avg'].agg(['mean', 'median'])
df['pollutant_avg'] = df.groupby('pollutant_id')['pollutant_avg'].transform(
    lambda x: x.fillna(x.median())
)

In [23]:
print("pollutant_avg:", df['pollutant_avg'].isnull().sum())

pollutant_avg: 0


TASK 3:

In [43]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


TASK 4:

In [25]:
df['state'].unique()


array(['Andhra_Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar',
       'Andhra Pradesh', 'Chandigarh', 'Chhattisgarh', 'Delhi', 'delhi',
       'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu_and_Kashmir',
       'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh',
       'Maharashtra', 'Meghalaya', 'Nagaland', 'Odisha', 'Rajasthan',
       'Punjab', 'Tamil-Nadu', 'TamilNadu', 'Tamil_Nadu', 'Telangana',
       'Uttar_Pradesh', 'Uttar Pradesh', 'Tripura', 'Uttarakhand',
       'West_Bengal', 'Andhra pradesh', 'Jammu & Kashmir', 'Puducherry',
       'Sikkim', 'Arunachal_Pradesh', 'MadhyaPradesh', 'Madhya_Pradesh',
       'Madhya Prades', 'UttarPradesh', 'Jammu and Kashmir',
       "Maharashtra'", 'Tamil Nadu', 'WestBengal', 'Madhya-Pradesh',
       'maharashtra', 'West Bengal'], dtype=object)

In [41]:
state_mapping = {
    "TamilNadu": "Tamil Nadu",
    "Tamil_Nadu": "Tamil Nadu",
    "Tamil-Nadu": "Tamil Nadu",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "J&K": "Jammu and Kashmir",
    "Jammu_and_Kashmir": "Jammu and Kashmir",
    "Jammu And Kashmir": "Jammu and Kashmir",
    "Uttar_Pradesh": "Uttar Pradesh",
    "UttarPradesh": "Uttar Pradesh",
    "MadhyaPradesh": "Madhya Pradesh",
    "WestBengal": "West Bengal",
    "West_Bengal": "West Bengal",
    "delhi": "Delhi",
    "maharashtra": "Maharashtra",
    "Maharashtra'": "Maharashtra",
    "Arunachal_Pradesh": "Arunachal Pradesh",
    "Andhra pradesh": "Andhra Pradesh",
    "Andhra_Pradesh": "Andhra Pradesh",
    "Madhya Prades": "Madhya Pradesh",
    "Madhya-Pradesh": "Madhya Pradesh",
    "Madhya_Pradesh": "Madhya Pradesh"
}

df.loc[:, 'state'] = df['state'].replace(state_mapping)
df.loc[:, 'state'] = df['state'].str.strip().str.title()

print(df['state'].unique())


['Andhra Pradesh' 'Arunachal Pradesh' 'Assam' 'Bihar' 'Chandigarh'
 'Chhattisgarh' 'Delhi' 'Gujarat' 'Haryana' 'Himachal Pradesh'
 'Jammu And Kashmir' 'Jharkhand' 'Karnataka' 'Kerala' 'Madhya Pradesh'
 'Maharashtra' 'Meghalaya' 'Nagaland' 'Odisha' 'Rajasthan' 'Punjab'
 'Tamil Nadu' 'Telangana' 'Uttar Pradesh' 'Tripura' 'Uttarakhand'
 'West Bengal' 'Puducherry' 'Sikkim']


in the above exercise i first checked all unique states then manually created a mapping of all variants of a particular state to one name only. then i reassigned it to all states.

In [ ]:
print(df['pollutant_id'].unique())

pollutant_mapping = {
    "OZONE": "Ozone",
    "O3": "Ozone"
}

df.loc[:, 'pollutant_id'] = df['pollutant_id'].replace(pollutant_mapping)

print(df['pollutant_id'].unique())

['PM2.5' 'O3' 'PM10' 'OZONE']
['PM2.5' 'Ozone' 'PM10']


i have shown a before and after here for better comparision

TASK 5:

In [33]:
pm25 = df[df['pollutant_id'] == 'PM2.5']['pollutant_avg']
Q1 = pm25.quantile(0.25)   # 25th percentile
Q3 = pm25.quantile(0.75)   # 75th percentile
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = pm25[(pm25 < lower_bound) | (pm25 > upper_bound)]
print("Outliers (IQR method):")
print(outliers)

Outliers (IQR method):
52      122.0
225      86.0
230      80.0
306     121.0
463      76.0
522     102.0
546      82.0
548      76.0
556      76.0
734      80.0
774     123.0
820     121.0
848      88.0
853      74.0
891      93.0
915      81.0
1074     76.0
1141     76.0
1163     90.0
1168     88.0
1303     74.0
1342     88.0
Name: pollutant_avg, dtype: float64


TASK 6:

In [45]:
df.loc[:, 'latitude'] = df['latitude'].round(4)
df.loc[:, 'longitude'] = df['longitude'].round(4)

print(df.loc[:, ['latitude', 'longitude']].head())

   latitude  longitude
0   13.6700    79.3500
1   13.6700    79.3500
2   13.6154    79.4092
3   16.5361    80.5942
4   27.1034    93.6796


In [ ]:
print(df.dtypes)

country           object
state             object
city              object
date_time         object
latitude         float64
longitude        float64
pollutant_id      object
pollutant_avg    float64
dtype: object


In [46]:
df.loc[:, 'date_time'] = pd.to_datetime(df['date_time'], dayfirst=True, errors='coerce')
df.loc[:, 'date_time'] = df['date_time'].dt.strftime('%d-%m-%Y %H:%M')

In [47]:
df.loc[:, 'date_time'] = pd.to_datetime(df['date_time'], dayfirst=True, errors='coerce')
df.loc[:, 'observation_date'] = df['date_time'].dt.date